# spaCy + FinBERT Quantitative Extraction Pipeline

## Setup

In [1]:
import os
import re
import pandas as pd
import spacy
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from pathlib import Path
from tqdm import tqdm

# Load models
nlp = spacy.load("en_core_web_md")
tokenizer = AutoTokenizer.from_pretrained("yiyanghkust/finbert-tone")
finbert = AutoModelForSequenceClassification.from_pretrained("yiyanghkust/finbert-tone")

## File Paths

In [2]:
TRANSCRIPT_DIR = Path(r"C:\Users\phata\OneDrive - George Mason University - O365 Production\AIT626\05. Projects\Final Project\Earnings2Insights\Transcripts")
OUTPUT_FILE = r"C:\Users\phata\OneDrive - George Mason University - O365 Production\AIT626\05. Projects\Final Project\Earnings2Insights\earningsExtractionSummary.csv"

## Helper Functions

In [3]:
def normalize_value(text):
    """Convert financial expressions to numeric floats including B/M/K and basis points."""
    if not isinstance(text, str) or not re.search(r'\d', text):
        return None

    # Remove commas and whitespace
    text_clean = text.replace(",", "").strip()

    # Extract all numeric substrings (ignore pure dots or empty strings)
    range_match = [s for s in re.findall(r"[\d\.]+", text_clean) if re.search(r"\d", s)]
    if not range_match:
        return None

    try:
        if len(range_match) == 2:
            num = sum(map(float, range_match)) / 2
        else:
            num = float(range_match[0])
    except ValueError:
        return None

    # Multipliers
    if re.search(r"billion|B", text_clean, re.I):
        num *= 1e9
    elif re.search(r"million|M", text_clean, re.I):
        num *= 1e6
    elif re.search(r"thousand|K", text_clean, re.I):
        num *= 1e3
    elif re.search(r"bps|basis points?", text_clean, re.I):
        num /= 10000  # basis points → percent

    return num



def finbert_sentiment(sentence):
    """Run FinBERT sentiment classification on a sentence."""
    inputs = tokenizer(sentence, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = finbert(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        label = torch.argmax(probs, dim=1).item()
    return ["negative", "neutral", "positive"][label]


def mark_QA_sections(text):
    lowered = text.lower()
    qa_start = None
    for marker in ["question and answer", "q&a", "operator:"]:
        match = lowered.find(marker)
        if match != -1:
            qa_start = match
            break
    return qa_start

In [4]:
METRIC_KEYWORDS = {
    "EPS": ["eps", "earnings per share", "diluted earnings", "basic earnings"],
    "REVENUE": ["revenue", "sales", "top line", "total revenue"],
    "MARGIN": ["margin", "gross margin", "operating margin", "profit margin", "ebitda margin"],
    "CASH_FLOW": ["cash flow", "operating cash", "free cash flow", "fcf"],
}

TIME_INDICATORS = {
    "PRIOR": ["prior", "last", "previous", "ago", "year-over-year", "yoy", "q/q"],
    "GUIDANCE": ["expect", "guidance", "forecast", "outlook", "projected"],
    "ESTIMATE": ["consensus", "analyst", "street"],
    "BEAT": ["beat", "exceeded", "outperform", "above"],
    "MISS": ["miss", "below", "underperform", "short of"]
}



DIRECTION_KEYWORDS = {
    "UP": ["up", "increased", "grew", "higher", "rose", "improved", "expanded"],
    "DOWN": ["down", "declined", "decreased", "lower", "fell", "reduced", "contracted"]
}


# DIRECTION CLASSIFIER (NEW)
def classify_direction(sentence, context):
    combined = (sentence + " " + context).lower()
    for direction, words in DIRECTION_KEYWORDS.items():
        if any(word in combined for word in words):
            return direction
    return None



# METRIC CLASSIFIER
def classify_metric(sentence, context):
    combined = (sentence + " " + context).lower()
    metric_type = "OTHER"
    metric_subtype = None
    time_indicator = "CURRENT"

    # Time indicators
    for key, words in TIME_INDICATORS.items():
        if any(w in combined for w in words):
            time_indicator = key
            break

    # Metric detection
    for metric, keywords in METRIC_KEYWORDS.items():
        if any(k in combined for k in keywords):
            metric_type = metric

            # Subtypes
            if metric == "EPS":
                if "diluted" in combined:
                    metric_subtype = "DILUTED"
                elif "basic" in combined:
                    metric_subtype = "BASIC"
                elif "gaap" in combined:
                    metric_subtype = "GAAP"
                elif "non-gaap" in combined or "adjusted" in combined:
                    metric_subtype = "NON_GAAP"

            elif metric == "REVENUE":
                if "total" in combined:
                    metric_subtype = "TOTAL"
                elif "segment" in combined or "product" in combined:
                    metric_subtype = "SEGMENT"

            elif metric == "MARGIN":
                if "gross" in combined:
                    metric_subtype = "GROSS"
                elif "operating" in combined:
                    metric_subtype = "OPERATING"
                elif "net" in combined or "profit" in combined:
                    metric_subtype = "NET"
                elif "ebitda" in combined:
                    metric_subtype = "EBITDA"

            break

    return metric_type, metric_subtype, time_indicator



# EXTRACT QUANTITATIVE CONTEXTS
def extract_quantitative_contexts(text):
    qa_start = mark_QA_sections(text)
    doc = nlp(text)
    records = []

    for sent in doc.sents:
        sent_text = sent.text.strip()

        # Skip non-financial sentences
        if not re.search(r"\d|%|\$|bps|million|billion", sent_text, re.I):
            continue
        if not any(k in sent_text.lower()
                   for kws in METRIC_KEYWORDS.values()
                   for k in kws):
            continue

        qa_flag = qa_start is not None and sent.start_char >= qa_start

        for ent in sent.ents:
            if ent.label_ not in ["MONEY", "PERCENT", "QUANTITY", "CARDINAL"]:
                continue

            # Local context window
            start_idx = max(ent.start - 3, 0)
            end_idx = min(ent.end + 3, len(sent))
            window_text = " ".join([t.text for t in sent[start_idx:end_idx]])

            metric_type, metric_subtype, time_indicator = classify_metric(
                sent_text, window_text
            )

            if metric_type != "OTHER":
                direction = classify_direction(sent_text, window_text)

                records.append({
                    "sentence": sent_text,
                    "context": window_text,
                    "entity_label": ent.label_,
                    "raw_value": ent.text,
                    "normalized_value": normalize_value(ent.text),
                    "metric_type": metric_type,
                    "metric_subtype": metric_subtype,
                    "time_indicator": time_indicator,
                    "direction": direction,
                    "QA": qa_flag
                })
    return records

# FILTER TO REVENUE + EPS (NEW)
def filter_revenue_eps(records):
    """
    Keep only REVENUE and EPS, each tagged with direction (UP/DOWN/None).
    """
    return [
        r for r in records
        if r["metric_type"] in ["REVENUE", "EPS"]
    ]


## Process Transcripts

In [5]:
all_records = []

for file in tqdm(os.listdir(TRANSCRIPT_DIR)):
    if not file.endswith(".txt"):
        continue
    filepath = TRANSCRIPT_DIR / file
    text = filepath.read_text(encoding="utf-8")
    text = re.sub(r"\s+", " ", text)

    # Extract ticker, quarter, year from filename
    parts = file.replace(".txt", "").split("_")
    ticker = parts[0]
    quarter, year = None, None
    if len(parts) >= 3:
        q_match = re.search(r"q(\d)", parts[1].lower())
        if q_match:
            quarter = int(q_match.group(1))
        y_match = re.search(r"\d{4}", parts[2])
        if y_match:
            year = int(y_match.group(0))

    # Extract quantitative records
    spacy_records = extract_quantitative_contexts(text)
    for r in spacy_records:
        r["Ticker"] = ticker
        r["Quarter"] = quarter
        r["Year"] = year
        r["sentiment"] = finbert_sentiment(r["sentence"])
    all_records.extend(spacy_records)

100%|██████████| 100/100 [07:35<00:00,  4.55s/it]


## Save & Export

In [10]:
df = pd.DataFrame(all_records)[[
    "Ticker", "Year", "Quarter", "QA", 
    "metric_type", "metric_subtype", "time_indicator",
    "direction",
    "raw_value", "normalized_value", "entity_label",
    "context", "sentiment", "sentence"
]]

df_filtered = df[
    (df["QA"] == False) &
    (df["metric_type"].isin(["REVENUE", "EPS"]))
].copy()

df_filtered = df_filtered.sort_values(
    ["Ticker", "Year", "Quarter", "metric_type", "direction"]
)

df_filtered.to_csv(OUTPUT_FILE, index=False)

print(f"Extracted dataset saved to: {OUTPUT_FILE}")
print(f"Total records: {len(df_filtered)}")
print("\nMetric type distribution:")
print(df_filtered["metric_type"].value_counts())
print("\nSample records:")
print(df_filtered.head(10))

Extracted dataset saved to: C:\Users\phata\OneDrive - George Mason University - O365 Production\AIT626\05. Projects\Final Project\Earnings2Insights\earningsExtractionSummary.csv
Total records: 2406

Metric type distribution:
metric_type
REVENUE    2035
EPS         371
Name: count, dtype: int64

Sample records:
   Ticker  Year  Quarter     QA metric_type metric_subtype time_indicator  \
6     ABM  2021        3  False         EPS       NON_GAAP          PRIOR   
7     ABM  2021        3  False         EPS       NON_GAAP          PRIOR   
8     ABM  2021        3  False         EPS       NON_GAAP          PRIOR   
0     ABM  2021        3  False         EPS       NON_GAAP        CURRENT   
9     ABM  2021        3  False         EPS       NON_GAAP       GUIDANCE   
2     ABM  2021        3  False     REVENUE        SEGMENT          PRIOR   
12    ABM  2021        3  False     REVENUE        SEGMENT          PRIOR   
13    ABM  2021        3  False     REVENUE        SEGMENT          PRIO

In [22]:
counts = df_filtered.groupby(
    ['Ticker', 'Quarter', 'Year', 'direction']
).size().reset_index(name='count')

winner = (
    counts.sort_values('count', ascending=False)
          .groupby(['Ticker', 'Quarter', 'Year'])
          .first()
          .reset_index()
)
winner.to_csv(r"C:\Users\phata\OneDrive - George Mason University - O365 Production\AIT626\05. Projects\Final Project\Earnings2Insights\direction.csv", index=False)